# Módulo 2 de Analítica de Datos — Manipulaciones con Apache Spark

En este módulo trabajaremos con **Apache Spark** dentro de **Databricks** para ingerir, limpiar, transformar y agregar datos provenientes de varios ficheros CSV.

### ¿Qué es Apache Spark?
Apache Spark es un motor de procesamiento distribuido que permite ejecutar cómputo en paralelo sobre clústeres de máquinas. En Databricks, Spark se ejecuta sobre el *cluster* (o *SQL warehouse* / *serverless compute*) asociado al notebook. Las operaciones se expresan como **DataFrames** (estructuras tabulares distribuidas similares a un `pandas.DataFrame`, pero particionadas y evaluadas de forma perezosa).

### Conceptos clave que veremos
- **Unity Catalog**: capa de gobernanza de datos en Databricks que organiza los activos en una jerarquía `catalog → schema (database) → table/volume/function/model`.
- **Volumes**: ubicación gobernada para almacenar **archivos no tabulares** (CSV, JSON, imágenes, modelos, etc.).
- **DataFrame API** de PySpark: `select`, `withColumn`, `groupBy`, `agg`, `dropDuplicates`, etc.
- **Esquemas explícitos** (`StructType`) frente a inferencia automática.
- **Evaluación perezosa (lazy evaluation)**: las transformaciones no se ejecutan hasta que aparece una *acción* (como `display`, `count`, `show`, `write`).

> **Nota**: los comandos `%sql`, `%python`, `%scala`, `%r` y `%md` son *magic commands* de Databricks que permiten cambiar de lenguaje dentro de una misma celda.

## 1. Carga de ficheros CSV en un Volume de Unity Catalog

Antes de poder leer datos con Spark, necesitamos un sitio donde dejarlos. En Databricks moderno, ese sitio recomendado para ficheros no tabulares es un **Volume** dentro de Unity Catalog.

### ¿Por qué un Volume y no DBFS?
Históricamente Databricks utilizaba **DBFS** (Databricks File System) y rutas tipo `/dbfs/...` o `dbfs:/...`. Hoy, **Unity Catalog Volumes** es la opción recomendada porque:

- Está **gobernada por Unity Catalog**: permisos `READ VOLUME` / `WRITE VOLUME` se gestionan con `GRANT`/`REVOKE`.
- Tiene **trazabilidad (lineage)** y auditoría integradas.
- Soporta **rutas POSIX** (`/Volumes/<catalog>/<schema>/<volume>/...`) accesibles desde Python (`open`), shell (`%sh`) y Spark.
- Es la opción soportada en **clústeres de acceso compartido (Shared)** y *serverless*, donde DBFS está restringido.

### Tipos de Volume
- **Managed volume**: Databricks gestiona la ubicación de almacenamiento (recomendado para empezar).
- **External volume**: apunta a una ruta de almacenamiento en la nube ya existente (S3, ADLS, GCS); útil para reutilizar datos que ya viven fuera de Databricks.

### Sintaxis general
```sql
CREATE VOLUME [IF NOT EXISTS] <catalog>.<schema>.<volume_name>
    [COMMENT '<descripción>'];
```
Si no se especifican `<catalog>` y `<schema>`, se usan los actuales (los puedes consultar con `SELECT current_catalog()`, `SELECT current_schema()`).

In [ ]:
%sql
-- Creamos un volumen llamado 'spark_lab' en el catálogo y schema actuales.
-- IF NOT EXISTS evita errores si ya se ha creado en una ejecución anterior (idempotente).
CREATE VOLUME IF NOT EXISTS spark_lab

### Descarga de los ficheros CSV de ejemplo

A continuación descargamos tres ficheros (`2019_edited.csv`, `2020_edited.csv`, `2021_edited.csv`) desde un repositorio público de GitHub y los guardamos en el Volume.

Aspectos a tener en cuenta:

- Usamos `spark.sql("SELECT current_catalog()")` para resolver dinámicamente el catálogo actual: así el notebook funciona en distintos workspaces sin tener que cambiar la ruta a mano.
- La ruta POSIX de un Volume tiene el formato `/Volumes/<catalog>/<schema>/<volume>/...`.
- `response.raise_for_status()` lanza una excepción si la descarga HTTP devuelve un error (4xx/5xx); es una buena práctica para detectar fallos de red pronto.
- Abrimos el fichero en modo binario (`"wb"`) porque `response.content` devuelve `bytes`.

**Alternativas opcionales**:
- Para descargas masivas o paralelas, podríamos usar `concurrent.futures.ThreadPoolExecutor`.
- Si los datos vivieran ya en S3/ADLS/GCS, no haría falta descargarlos: bastaría con leerlos directamente o crear un *external volume* sobre esa ruta.

In [ ]:
import requests

# Resolvemos el catálogo actual para construir la ruta del Volume de forma dinámica.
# Esto hace el notebook portable entre entornos (dev/test/prod) que tengan distintos catálogos.
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Ruta base del Volume usando el formato POSIX /Volumes/<catalog>/<schema>/<volume>.
# 'default' es el schema por defecto; ajústalo si has cambiado de schema con USE SCHEMA.
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# Lista de ficheros que vamos a descargar (datos de ventas de 2019, 2020 y 2021).
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

# Descargamos cada fichero del repositorio público y lo escribimos en el Volume.
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()  # Si la respuesta HTTP no es 2xx, lanza excepción.

    # 'wb' = write binary. response.content es bytes, así evitamos problemas de codificación.
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

## 2. Lectura de los ficheros CSV en un DataFrame

Una vez que los ficheros están en el Volume, los leemos con la API de Spark. Spark soporta lectura paralela de múltiples ficheros mediante **patrones glob** (`*`, `?`, `[...]`).

### Sintaxis general
```python
df = spark.read.load(<ruta>, format=<formato>, **opciones)
# equivalente:
df = spark.read.format("csv").load(<ruta>)
df = spark.read.csv(<ruta>)
```

### Formatos soportados de fábrica
`csv`, `json`, `parquet`, `orc`, `text`, `avro`, `delta`, `xml` (a partir de DBR 14.3), `binaryFile`.

### Opciones útiles para CSV (no se aplican aquí, pero son habituales)
- `header=True` → la primera fila contiene los nombres de columna.
- `inferSchema=True` → Spark deduce los tipos leyendo los datos (lento: hace una pasada extra).
- `sep=","` → separador de campos (también `delimiter`).
- `quote='"'`, `escape='\\'` → manejo de comillas y caracteres de escape.
- `nullValue="NA"` → texto que se interpretará como `NULL`.
- `mode="PERMISSIVE" | "DROPMALFORMED" | "FAILFAST"` → comportamiento ante filas mal formadas.
- `multiLine=True` → permite que un campo contenga saltos de línea.

### `display()` vs `show()`
- `display(df)` es **propia de Databricks**: pinta una tabla interactiva con paginación, descarga y permite construir gráficos sin código.
- `df.show(n)` es estándar de Spark y solo imprime texto en la salida.
- `df.limit(100)` limita el resultado a 100 filas; es buena práctica al explorar datasets grandes.

In [ ]:
# El patrón '*_edited.csv' lee de golpe los tres ficheros (2019, 2020, 2021).
# Spark unifica los esquemas si son compatibles y procesa los ficheros en paralelo.
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv')

# Mostramos solo 100 filas para no sobrecargar la UI ni transferir datos innecesarios al driver.
display(df.limit(100))

## 3. Definición explícita del esquema del DataFrame

En la lectura anterior no especificamos esquema, por lo que Spark trató **todas las columnas como `string`** y les asignó nombres genéricos (`_c0`, `_c1`, ...). Para trabajar correctamente con tipos numéricos, fechas, etc., conviene definir un esquema.

### Tres opciones para gestionar el esquema
1. **Esquema inferido** (`inferSchema=True`): cómodo, pero hace una pasada extra sobre los datos y puede equivocarse con tipos ambiguos.
2. **Esquema explícito** (`StructType` + `StructField`): la opción **recomendada en producción**: rápido, predecible y autodocumentado.
3. **Schema DDL string**: forma compacta como `"SalesOrderNumber STRING, SalesOrderLineNumber INT, ..."`.

### Tipos de datos comunes en `pyspark.sql.types`
| Tipo Spark | Descripción |
|---|---|
| `StringType` | Cadena de caracteres |
| `IntegerType` / `LongType` | Entero 32 / 64 bits |
| `FloatType` / `DoubleType` | Coma flotante 32 / 64 bits |
| `DecimalType(p,s)` | Decimal exacto (recomendado para dinero) |
| `BooleanType` | True/False |
| `DateType` | Fecha (sin hora) |
| `TimestampType` | Fecha y hora |
| `ArrayType(elementType)` | Lista |
| `MapType(keyType, valueType)` | Diccionario |
| `StructType([StructField(...)])` | Estructura anidada |

> **Consideración opcional**: para columnas monetarias, `DecimalType(10, 2)` evita errores de redondeo de coma flotante. Aquí usamos `FloatType` por seguir el ejemplo original.

### Sobre los imports `from ... import *`
Aunque `from pyspark.sql.types import *` y `from pyspark.sql.functions import *` son cómodos en notebooks, en código de producción es mejor importar solo lo que se necesita (`from pyspark.sql.functions import col, year, split`) para evitar colisiones de nombres.

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Definimos el esquema describiendo cada columna con su tipo.
# El orden DEBE coincidir con el orden de las columnas en el CSV.
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),     # Identificador del pedido
    StructField("SalesOrderLineNumber", IntegerType()), # Número de línea dentro del pedido
    StructField("OrderDate", DateType()),               # Fecha del pedido
    StructField("CustomerName", StringType()),          # Nombre completo del cliente
    StructField("Email", StringType()),                 # Correo electrónico
    StructField("Item", StringType()),                  # Producto vendido
    StructField("Quantity", IntegerType()),             # Unidades vendidas
    StructField("UnitPrice", FloatType()),              # Precio unitario
    StructField("Tax", FloatType())                     # Impuesto aplicado
])

# Volvemos a leer los CSV pasando el esquema. Ahora cada columna tendrá su tipo correcto
# y su nombre real, sin que Spark haga una pasada de inferencia.
df = spark.read.load(
    f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv',
    format='csv',
    schema=orderSchema
)

display(df.limit(100))

## 4. Limpieza de datos

Antes de hacer análisis, conviene aplicar una capa de **calidad de datos**. Las operaciones más típicas son:

- **Eliminar duplicados**: `df.dropDuplicates()` (o `df.dropDuplicates(["col1", "col2"])` para deduplicar por ciertas columnas).
- **Tratar nulos**: `df.dropna()`, `df.fillna(valor)`, `df.na.replace(...)`.
- **Filtrar filas atípicas**: `df.filter(col("x") > 0)`.
- **Recalcular o reemplazar columnas**: `df.withColumn("nueva", expresión)` (si la columna ya existe, la sobrescribe).
- **Castear tipos**: `col("x").cast("float")`.

### `withColumn` y la inmutabilidad
Los DataFrames de Spark son **inmutables**: `withColumn` no modifica `df`, sino que devuelve **un nuevo DataFrame** con la transformación añadida. Por eso reasignamos `df = df.withColumn(...)`.

### Acerca de la evaluación perezosa
`dropDuplicates` y `withColumn` son **transformaciones**: no se ejecutan en este momento. Spark construye un **plan lógico** y solo lo materializa cuando aparece una **acción** (`display`, `count`, `collect`, `write`, ...). Esto le permite optimizar el plan globalmente (filtrar antes, reordenar joins, etc.).

### Caso de este ejemplo
Recalculamos la columna `Tax` como el 8 % del `UnitPrice`. Si los datos originales tuvieran impuestos inconsistentes, este paso los normaliza. Luego forzamos el tipo a `float` por consistencia.

> **Consideración opcional**: para dejar trazabilidad de la limpieza, en pipelines reales se suele guardar el resultado como una tabla **Delta Lake** (`df.write.format("delta").saveAsTable(...)`), lo que permite *time travel*, transacciones ACID y `OPTIMIZE`/`VACUUM`.

In [ ]:
from pyspark.sql.functions import col

# 1) Eliminamos filas idénticas (duplicados exactos en todas las columnas).
df = df.dropDuplicates()

# 2) Recalculamos la columna 'Tax' como el 8% del precio unitario.
#    withColumn devuelve un nuevo DataFrame; al reasignar, sustituimos la columna existente.
df = df.withColumn('Tax', col('UnitPrice') * 0.08)

# 3) Aseguramos que el tipo final de 'Tax' sea Float (la multiplicación puede devolver Double).
df = df.withColumn('Tax', col('Tax').cast("float"))

## 5. Creación de un nuevo DataFrame derivado (clientes)

Vamos a construir un DataFrame `customers_df` que contenga solo las columnas relevantes para clientes y, además, descomponga `CustomerName` en `FirstName` y `LastName`.

### Operaciones que se usan
- **`select(...)`**: proyecta (selecciona) un subconjunto de columnas. Equivale a `SELECT a, b, c FROM df` en SQL.
- **`split(col, pattern)`**: divide una cadena por un patrón (regex) y devuelve un `ArrayType(StringType)`.
- **`.getItem(i)`** (o el indexado `[i]`): accede al elemento `i` de un array.

### Limitaciones de este enfoque (importante)
Separar nombre y apellido por el primer espacio funciona en datos sencillos, pero **es frágil** con nombres reales:
- `"María José García"` → `FirstName="María"`, `LastName="José"` (se pierde "García").
- `"Juan de la Torre"` → solo conserva `"Juan"` y `"de"`.

**Alternativas más robustas**:
```python
# Tomar todo lo posterior al primer espacio como apellido
from pyspark.sql.functions import split, expr
df.withColumn("FirstName", split(col("CustomerName"), " ").getItem(0))
  .withColumn("LastName", expr("substring(CustomerName, instr(CustomerName, ' ') + 1)"))
```
O, si se conoce el formato, usar **regex** con `regexp_extract`.

In [ ]:
# Proyectamos las columnas que nos interesan para el análisis de clientes.
customers_df = df.select("CustomerName", "Email", "Item", "Quantity")

# split() devuelve un array; getItem(0) toma el primer elemento (nombre).
customers_df = customers_df.withColumn(
    "FirstName",
    split(customers_df["CustomerName"], " ").getItem(0)
)

# getItem(1) toma el segundo elemento (apellido). Ojo: si hay más de dos palabras, se pierden.
customers_df = customers_df.withColumn(
    "LastName",
    split(customers_df["CustomerName"], " ").getItem(1)
)

display(customers_df)

## 6. Filas distintas: `count()` vs `distinct().count()`

Una pregunta típica al explorar un DataFrame es: *¿cuántas filas tengo y cuántas son únicas?*

- **`df.count()`**: cuenta todas las filas (incluyendo repetidas).
- **`df.distinct().count()`**: cuenta filas **únicas** considerando todas las columnas seleccionadas.

Si el segundo número es menor que el primero, hay duplicados.

### Coste de estas operaciones
Ambas son **acciones** y disparan la ejecución del plan completo. `distinct()` además requiere un *shuffle* (intercambio de datos entre nodos), por lo que es más caro que un simple `count`.

### Variantes útiles
- `df.dropDuplicates(["Email"]).count()` → clientes únicos por email.
- `df.select("Email").distinct().count()` → equivalente, más explícito.
- `df.summary()` o `df.describe()` → estadísticas rápidas de columnas numéricas.
- `df.printSchema()` → muestra el esquema y los tipos.

In [ ]:
# Total de filas en customers_df (incluye repetidas).
print(customers_df.count())

# Filas únicas (combinación distinta de CustomerName, Email, Item, Quantity, FirstName, LastName).
print(customers_df.distinct().count())

## 7. Agregaciones: ventas por producto

Las **agregaciones** resumen muchas filas en una sola por grupo. La fórmula básica es:

```python
df.groupBy(<columnas_de_grupo>).<función_agregación>()
```

### Funciones de agregación habituales (en `pyspark.sql.functions`)
- `sum`, `avg` (= `mean`), `min`, `max`, `count`, `countDistinct`
- `stddev`, `variance`, `collect_list`, `collect_set`
- `first`, `last`, `approx_count_distinct`

### `groupBy(...).sum()` vs `groupBy(...).agg(...)`
- `groupBy(...).sum()` aplica `sum` a **todas las columnas numéricas** del DataFrame seleccionado y crea columnas con prefijo `sum(...)`.
- `groupBy(...).agg(...)` permite control fino: aplicar varias funciones, renombrar con `alias`, etc.

Ejemplo equivalente con `agg`:
```python
from pyspark.sql.functions import sum as _sum
df.groupBy("Item").agg(_sum("Quantity").alias("TotalQuantity"))
```

> **Consideración opcional**: si vas a hacer muchas agregaciones sobre el mismo DataFrame, considera **cachearlo** con `df.cache()` (o `persist()`) para evitar releer los CSV en cada acción. Recuerda hacer `df.unpersist()` cuando ya no lo necesites.

In [ ]:
# Seleccionamos solo Item y Quantity, agrupamos por Item y sumamos las cantidades.
# Resultado: una fila por producto con la columna 'sum(Quantity)'.
productSales = df.select("Item", "Quantity").groupBy("Item").sum()
display(productSales)

## 8. Agregación temporal: ventas por año

Para analizar tendencias en el tiempo, es muy común agrupar por año, mes o día. PySpark trae funciones específicas para extraer partes de una fecha:

| Función | Devuelve |
|---|---|
| `year(col)` | Año (`int`) |
| `month(col)` | Mes 1-12 |
| `dayofmonth(col)` | Día del mes |
| `dayofweek(col)` | Día de la semana (1 = domingo) |
| `weekofyear(col)` | Número de semana ISO |
| `quarter(col)` | Trimestre 1-4 |
| `date_format(col, fmt)` | Formato libre, p. ej. `'yyyy-MM'` |

### `alias`, `orderBy` y otras transformaciones encadenadas
- **`alias("nombre")`** renombra una columna o expresión calculada (equivale a `AS` en SQL).
- **`orderBy("col")`** ordena el resultado; usa `desc("col")` o `col("x").desc()` para orden descendente.
- Encadenar transformaciones (`.select(...).groupBy(...).count().orderBy(...)`) es idiomático en PySpark y se lee de forma similar a SQL.

Equivalente en SQL:
```sql
SELECT year(OrderDate) AS Year, COUNT(*) AS cnt
FROM df
GROUP BY year(OrderDate)
ORDER BY Year;
```

> **Consideración opcional**: si quisieras visualizar esta serie temporal como gráfico de barras o líneas, en Databricks puedes ejecutar la celda y, sobre la salida de `display()`, pulsar el icono `+` para añadir una visualización (Bar/Line chart) sin escribir código adicional.

In [ ]:
# 1) Extraemos el año de OrderDate y lo renombramos a 'Year' con alias.
# 2) Agrupamos por Year y contamos cuántos pedidos hay en cada año.
# 3) Ordenamos cronológicamente por Year ascendente.
yearlySales = (
    df.select(year("OrderDate").alias("Year"))
      .groupBy("Year")
      .count()
      .orderBy("Year")
)
display(yearlySales)

## Resumen del módulo

En este notebook hemos cubierto:

1. **Unity Catalog Volumes**: creación con `CREATE VOLUME` y uso de rutas POSIX `/Volumes/...` para almacenar ficheros no tabulares de forma gobernada.
2. **Ingesta**: descarga HTTP con `requests` y escritura directa al Volume.
3. **Lectura con Spark**: `spark.read.load(...)` con globs para procesar varios CSV en paralelo.
4. **Esquemas**: `StructType` + `StructField` para tipar columnas explícitamente, evitando inferencia.
5. **Limpieza**: `dropDuplicates`, `withColumn`, `cast`.
6. **Transformaciones**: `select`, `split`, `getItem` para derivar nuevas columnas.
7. **Conteos**: `count` vs `distinct().count()`.
8. **Agregaciones**: `groupBy(...).sum()`, `groupBy(...).count()`, funciones temporales (`year`).

### Próximos pasos sugeridos
- Persistir resultados como **tabla Delta** (`df.write.format("delta").saveAsTable("ventas")`) para beneficiarse de transacciones ACID, *time travel* y rendimiento.
- Aprender **Spark SQL** y `spark.sql("SELECT ...")` como alternativa al DataFrame API.
- Explorar **Delta Live Tables (DLT)** y **Lakeflow** para construir pipelines declarativos.
- Profundizar en optimizaciones: *partition pruning*, `OPTIMIZE`, `Z-ORDER`, *caching*, *broadcast joins*.
- Revisar **Unity Catalog Lineage** en la UI para ver cómo se conectan tus tablas y notebooks.